# The cold-start problem — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def cosine(a,b):
    a=np.asarray(a,dtype=float); b=np.asarray(b,dtype=float)
    return float(a@b/(np.linalg.norm(a)*np.linalg.norm(b)))
def sigmoid(x): return 1/(1+np.exp(-np.asarray(x)))
def dcg(rels): return sum((2**r-1)/math.log2(i+2) for i,r in enumerate(rels))
def ndcg(rels):
    best=sorted(rels, reverse=True)
    return dcg(rels)/dcg(best) if dcg(best)>0 else 0.0
print('setup ok')

## Making useful recommendations before interaction history exists
Cold start is the missing-data failure mode of recommender systems. The examples compute a popularity prior, content fallback for a new item, onboarding preference elicitation, Bayesian shrinkage, and hybrid weighting that changes as evidence arrives.

In [ ]:
# 1) popularity prior is the safest first recommendation with no user history
clicks=np.array([50,20,5]); impressions=np.array([1000,200,20]); ctr=clicks/impressions
plt.figure(figsize=(6,3)); plt.bar(['A','B','C'],ctr); plt.title('raw popularity/CTR prior')
assert int(np.argmax(ctr))==2 and abs(ctr[2]-0.25)<1e-12

In [ ]:
# 2) Bayesian shrinkage prevents tiny-count items from dominating
clicks=np.array([50,20,5]); impressions=np.array([1000,200,20]); alpha=10; beta=190
post=(clicks+alpha)/(impressions+alpha+beta)
plt.figure(figsize=(6,3)); plt.bar(['A','B','C'],post); plt.title('shrunk CTR estimates')
assert int(np.argmax(post))==1 and abs(post[1]-0.075)<1e-12

In [ ]:
# 3) new item can still be scored from content attributes
profile=np.array([1,.5]); items=np.array([[1,0],[0,1],[1,1]],float); scores=np.array([cosine(profile,x) for x in items])
plt.figure(figsize=(6,3)); plt.bar(['old A','old B','new C'],scores); plt.title('content fallback for new item')
assert int(np.argmax(scores))==2 and abs(scores[2]-0.9486832980505138)<1e-12

In [ ]:
# 4) onboarding answers create an immediate user profile
answers=np.array([[1,0,0],[0,1,1]],float); likes=np.array([1,1]); profile=likes@answers/likes.sum()
plt.figure(figsize=(6,3)); plt.bar(['sports','jobs','video'],profile); plt.title('two questions -> profile')
assert np.allclose(profile,[0.5,0.5,0.5])

In [ ]:
# 5) hybrid evidence weight shifts from prior to personalization as events accumulate
events=np.arange(0,21); w=events/(events+5); prior=.06; personal=.18; score=(1-w)*prior+w*personal
plt.figure(figsize=(6,3)); plt.plot(events,score,'-o',ms=3); plt.xlabel('user events'); plt.ylabel('hybrid score'); plt.title('personalization ramps in')
assert abs(score[0]-0.06)<1e-12 and score[-1]>0.15